<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" alt="EDR|AI" width="300"/>

# Chapter 21 — Measurement and Operationalization

This is the **companion notebook** of [Chapter 21 — Measurement and Operationalization](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part4-credible-evidence/18-measurement-and-operationalization.html) from **EDR|AI — Evidence-Driven Research in the Age of AI**. Authored by [Davi Moreira](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html).

[Open the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part4-credible-evidence/18-measurement-and-operationalization.html) · [Book home](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html) · [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html)

*AI is your arm and your research assistant, not your brain.*

## How to use this notebook

1. Work top to bottom, with the chapter open in another tab.
2. Copy each **AI prompt** into your AI tool, run it, then record in the response cell what came back and what you verified.
3. Run the code cells; change something and run again.
4. Finish the **It is your turn** workspace at the end — that is this chapter's step of your own research project.
5. Log every AI use in your **AI Research Ledger**: task · tool · prompt · output summary · decision · verification method · remaining concern · you as the responsible researcher.
6. Your AI can be more than a chatbot: agentic tools can run multi-step work for you. Delegating boldly is fine; reviewing, curating, and deciding stay yours.

> **The research decision.** Decide how to turn the abstract idea your question is
> about into one concrete number you can record, repeat, and defend. Name the
> concept, name the narrower construct that will stand in for it, name the
> indicator you will actually measure, and then say plainly what that indicator
> captures and what it quietly leaves out.

## Code from the chapter

The cells below come from the chapter. Run them, then change something and run again — the numbers should move the way the chapter says they will.

*From the section “A worked example”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np, pandas as pd
SEED = 464
rng = np.random.default_rng(SEED)

n_per_site, sites = 60, 16
pilot = np.repeat([True] * 8 + [False] * 8, n_per_site)
site = np.repeat(np.arange(sites), n_per_site)

# The construct is willingness to recommend; the indicator is one 1-7 item.
site_effect = rng.normal(0, 0.25, size=sites)
person = (4.3 + 0.35 * pilot + site_effect[site]
          + rng.normal(0, 1.2, size=len(site)))          # people differ, stably
item = np.clip(np.round(person + rng.normal(0, 0.5, size=len(person))), 1, 7)

# Reliability: re-ask a random tenth two weeks later. Same PEOPLE, same item.
retest_idx = rng.choice(len(item), size=len(item) // 10, replace=False)
retest = np.clip(np.round(person[retest_idx]
                          + rng.normal(0, 0.5, size=len(retest_idx))), 1, 7)
r = np.corrcoef(item[retest_idx], retest)[0, 1]

print(pd.DataFrame({"pilot site": pilot, "item": item})
      .groupby("pilot site")["item"].agg(["size", "mean"]).round(2).to_string())
print(f"\ntest-retest correlation on the re-asked tenth : {r:.2f}")
print("that number says the item is repeatable. it says NOTHING about whether")
print("recommend-a-friend captures engagement, or about the people who quit")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “When the measurement is the contribution”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
SEED = 464
rng = np.random.default_rng(SEED)

n = 400                                            # frontline staff in the field test
engagement = rng.normal(0, 1, n)                   # the construct nobody observes
# Four new items, each = engagement + item-specific noise.
items = engagement[:, None] + rng.normal(0, 1.0, size=(n, 4))
new_scale = items.mean(axis=1)
# Convergent: an established engagement measure, read with its own noise.
established = engagement + rng.normal(0, 0.8, n)
# Discriminant: a different construct that should barely move with engagement.
commute = 0.1 * engagement + rng.normal(0, 1, n)

def r_ci(x, y):
    r = np.corrcoef(x, y)[0, 1]
    z, se = np.arctanh(r), 1 / np.sqrt(len(x) - 3)
    return r, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)

for label, other in [("with the established measure (convergent)", established),
                     ("with commute length (discriminant)      ", commute)]:
    r, lo, hi = r_ci(new_scale, other)
    print(f"new scale {label}: r = {r:.2f}  [95% interval {lo:.2f} to {hi:.2f}]")
print("\nthe pattern is evidence FOR one reading of the scale, in this workforce;")
print("it does not show that the scale measures engagement everywhere")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “Validating labels an AI produced”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
SEED = 464
rng = np.random.default_rng(SEED)

N, n_gold = 3_000, 300
# TRUTH: 18% of comments mention burnout. Nobody sees this column.
truth = rng.random(N) < 0.18
# The AI labeler misses 40% of real mentions and flags 3% of the rest.
ai = np.where(truth, rng.random(N) < 0.60, rng.random(N) < 0.03)

# You hand-code a RANDOM 300, blind to the AI label (assume your codes match truth).
gold = rng.choice(N, size=n_gold, replace=False)
h, a = truth[gold].astype(float), ai[gold].astype(float)

agree = (h == a).mean()
p_chance = h.mean() * a.mean() + (1 - h.mean()) * (1 - a.mean())
kappa = (agree - p_chance) / (1 - p_chance)
caught = a[h == 1].mean()                        # share of real mentions the AI flagged
false_flags = a[h == 0].mean()                   # share of non-mentions it flagged

naive = ai.mean()               # the AI labeled all 3,000: known exactly, not estimated
d = h - a                       # the AI's error on each coded comment
corrected = naive + d.mean()
# The only chance left is WHICH 300 you coded. The error average carries it,
# shrunk by the share of the pile you did not code (sampling without replacement).
fpc = 1 - n_gold / N
se = np.sqrt(fpc * d.var(ddof=1) / n_gold)
se_h = np.sqrt(fpc * h.var(ddof=1) / n_gold)    # your 300 codes on their own

print(f"agreement on the coded 300     : {agree:.2f}   kappa: {kappa:.2f}")
print(f"real mentions the AI caught    : {caught:.2f}   false flags: {false_flags:.2f}")
print(f"true share (hidden)            : {truth.mean():.3f}")
print(f"AI-only share                  : {naive:.3f}")
print(f"corrected with the coded 300   : {corrected:.3f}  "
      f"[95% interval {corrected - 1.96*se:.3f} to {corrected + 1.96*se:.3f}]")
print(f"your 300 codes alone           : {h.mean():.3f}  "
      f"[95% interval {h.mean() - 1.96*se_h:.3f} to {h.mean() + 1.96*se_h:.3f}]")
print("\nhigh agreement did not make the AI's share right;")
print("the random hand-coded subset is what lets you repair it, and bound it")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

## It is your turn

<!-- station-pointer:begin -->
> **You are working inside [Studio 6: Govern data and measurement](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio06-govern-data-measurement.html).**
> Keep what you write here; the studio's milestone chapter is
> where it joins the other lessons' pieces into one artifact
> you can defend.
<!-- station-pointer:end -->

*You know where your data come from. This step pins down what your numbers actually
mean, and what they do not.*

The hands-on half of this section lives in the chapter's **companion notebook**: open it in Colab with the badge at the top, and work the steps there.

Commit your own draft first, then delegate. Each prompt is a checkable job, not a
verdict to trust. Run them as a loop: the first answer names the obvious
instruments, and the useful pass is the second one, where you say which facet the
tool skipped and ask it to try again. Agentic tools will happily go three or four
rounds and return a polished measurement plan. Polish is not validity, and the
sentence naming what your number leaves out still has to be written by you.

> **Do not delegate.**
>
> Which **concept** your question is about, which **construct** honestly stands for it,
> and where your **indicator's** meaning must stop are yours alone. The tool can list
> instruments and name limitations, but only you decide that a recommend-a-friend
> rating is a fair stand-in for the facet of engagement you actually care about, and
> only you write the sentence that refuses to call one facet the whole concept. Naming
> what your number leaves out is the researcher's job, not the model's.

**Step 1.** List every concept your question contains and circle the abstract ones, the
words no instrument reads: engagement, health, participation, quality,
competitiveness. Those are the ones that need a ladder.

✍️ **Your work for step 1.** Double-click this cell and write your answer here.

**Step 2.** Under each circled concept write the construct you will really measure, plus one
line on why that facet and not a neighboring one. Choosing a facet is not
cheating. Pretending you measured the whole concept is.

✍️ **Your work for step 2.** Double-click this cell and write your answer here.

**Step 3.** Under each construct write the indicator: the exact procedure and the exact
number it produces, specific enough that a stranger could repeat it and get a
comparable value.

**Locate the standard indicators.**

```text
Act as a survey-methods assistant. I want to measure the construct "willingness to
recommend one's workplace" in an organizational study. Name the standard, published
instruments researchers use for this construct, with the instrument name, how it is
scored, and a real source I can open. Only list instruments you are confident
exist; mark anything uncertain.
```

After running, verify: open one named source and confirm the instrument exists and
measures what the tool says it does. Counters *confident fabrication* (an invented
scale name arrives as fluently as a real one).

✍️ **Your work for step 3.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 4.** Under each indicator finish this sentence in your own words: "this number does
not capture ___." Use the facet that worries you most, and add who your
measurement never reaches.

**List the limits, so you can verify them.**

```text
Here is my operationalization: concept = employee engagement, construct =
willingness to recommend the workplace, indicator = mean of a single 1-7 agreement
item. Return a table of every facet of engagement this indicator does NOT capture,
one row each, with why it matters and who it fails to hear from.
```

After running, verify: check the table against a published engagement framework you
retrieve yourself. Counters *illusion of completeness* (a tidy list that still omits
the facet that most threatens your claim).

✍️ **Your work for step 4.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 5.** Write one sentence saying what your number means and one saying what you will do
with it. Then name the strongest rival reading: what else could produce the same
number? That pair, plus the rival, is what your evidence has to support.

**Red-team the swap.**

```text
Act as a hostile reviewer. Here is my claim: "the pilot sites are more engaged,
because more people there would recommend the company." Name every place my
indicator fails to support the word "engaged." Do not rewrite the claim for me.
```

After running, verify: if it only praises the design, push back and demand the single
worst gap. Counters *sycophantic agreement* (praise that reviews your ego, not your
measurement).

✍️ **Your work for step 5.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 6.** Plan one reliability check that matches an error source you actually worry about,
then run it: the same units measured twice (occasions), two coders on the same
material (raters), or two halves of your *items* scored for the same people
(split-half). Do not compare two halves of your respondents and call it
reliability. That measures how a group average moves between samples of people,
which is a different question. If your indicator is a single item measured once,
split-half is unavailable and the other two may be too; "no defensible
reliability check is available for this measure and use" is an honest finding.
Write it down, and then either narrow what you claim or choose a different
instrument.

✍️ **Your work for step 6.** Double-click this cell and write your answer here.

**Step 7.** Log the step in your AI Research Ledger, and verify at least one measure with a
named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html).
Primary-source reading fits this chapter: find the published instrument or
method behind your indicator and open it. What you get there is evidence, not a
verdict. Published validation was done on some population, for some use; check
how close those are to yours, and treat the distance as part of your claim
boundary. An AI reviewer may run the check with you; the
decision to accept or reject stays yours.

<!-- studio-continue:begin -->
> **Milestone next.** This was the last lesson of Studio 6.
> [Milestone 6: Your data and measurement, governed](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/milestone06-govern-data-measurement.html#milestone) is where
> the lessons' pieces become the studio's versioned artifact.
> Produce it before you move on.
<!-- studio-continue:end -->

✍️ **Your work for step 7.** Double-click this cell and write your answer here.

### The standard this section is held to

Use this as a self-check while you work. It is also the bar the same work meets later, once your project carries it. Each row: **0** missing, **1** attempted but incomplete, generic, or unverified, **2** complete, specific to your own project, and verified where a check applies. **16 points in all.**

| # | Criterion | 0–2 |
|---|---|---|
| Step 1 | List every concept your question contains and circle the abstract ones, the words no instrument reads: engagement, health, participation, quality,… | |
| Step 2 | Under each circled concept write the construct you will really measure, plus one line on why that facet and not a neighboring one | |
| Step 3 | Under each construct write the indicator: the exact procedure and the exact number it produces, specific enough that a stranger could repeat it and… | |
| Step 4 | Under each indicator finish this sentence in your own words: "this number does not capture ___." Use the facet that worries you most, and add who… | |
| Step 5 | Write one sentence saying what your number means and one saying what you will do with it | |
| Step 6 | Plan one reliability check that matches an error source you actually worry about, then run it: the same units measured twice (occasions), two coders… | |
| Step 7 | Log the step in your AI Research Ledger, and verify at least one measure with a named method from the Verification Guide | |
| + | Craft and verification record: AI use logged in your AI Research Ledger, claims stated with their uncertainty, and each key claim verified with a named method | |

In [ ]:
# Scratch space — use this cell for any code your steps need.

**Before you leave this notebook:** add today's rows to your AI Research Ledger, and verify your key claim with a named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). AI can review AI — but the last decision is human.

Next: [Chapter 22 — AI as Programmer](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part4-credible-evidence/19-ai-as-programmer.html).